# 2. Binning and WoE

Each feature is grouped into a few bins, and each bin gets a WoE value that says how risky its loans are compared with the average. These WoE values are what the model uses later. The bins are found separately for the 36-month and the 60-month loans, using their train sets only.

In [18]:
from pathlib import Path
import pandas as pd
from optbinning import BinningProcess

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

train_36 = pd.read_parquet(PROCESSED / "train_36.parquet")
train_60 = pd.read_parquet(PROCESSED / "train_60.parquet")

### 2.1 The features

Every column except the target is a feature. They come in two kinds. Number features, like income, are binned by ranges of values. Category features, like home ownership, are binned by their categories.

In [19]:
features = [c for c in train_36.columns if c != "target"]
numbers = [c for c in features if pd.api.types.is_numeric_dtype(train_36[c])]
categories = [c for c in features if c not in numbers]

print("Category features:", categories)
print("Number features:", numbers)

Category features: ['purpose', 'verification_status', 'home_ownership']
Number features: ['annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec', 'revol_bal', 'revol_util', 'fico', 'credit_history_years', 'loan_to_income', 'emp_length_years']


### 2.2 Bins for the 36-month loans

The bins are found with optbinning, a Python library for credit scoring. It searches for the best bins under three rules. Each bin holds at least 5% of the loans, each has both good and bad loans, and WoE goes in one direction. Missing values always get their own bin.

In [20]:
binning_36 = BinningProcess(
    variable_names=features,
    categorical_variables=categories,
    min_bin_size=0.05,
    binning_fit_params={c: {"monotonic_trend": "auto_asc_desc"} for c in features},
)
binning_36.fit(train_36[features], train_36["target"])

binning_36.summary()[["name", "dtype", "n_bins"]]

,name,dtype,n_bins
0,purpose,categorical,4
1,verification_status,categorical,3
2,home_ownership,categorical,3
3,annual_inc,numerical,11
4,dti,numerical,13
5,delinq_2yrs,numerical,3
6,inq_last_6mths,numerical,4
7,open_acc,numerical,5
8,total_acc,numerical,8
9,pub_rec,numerical,2


For example, for Fico, these are the bins:

In [21]:
table = binning_36.get_binned_variable("fico").binning_table.build()
table[["Bin", "Count", "Event", "Event rate", "WoE"]]

,Bin,Count,Event,Event rate,WoE
0,"(-inf, 664.50)",28076,5213,0.185675,-0.417323
1,"[664.50, 669.50)",28307,4959,0.175186,-0.34638
2,"[669.50, 674.50)",28744,4919,0.171131,-0.318057
3,"[674.50, 679.50)",26130,4088,0.156449,-0.210793
4,"[679.50, 684.50)",26157,4031,0.154108,-0.192948
5,"[684.50, 689.50)",23572,3401,0.144281,-0.115511
6,"[689.50, 694.50)",22721,3008,0.132389,-0.015684
7,"[694.50, 699.50)",20632,2664,0.129120,0.013077
8,"[699.50, 709.50)",35136,4040,0.114982,0.145147
9,"[709.50, 719.50)",26746,2655,0.099267,0.309707


### 2.3 Bins for the 60-month loans

The same steps are repeated for the 60-month loans. Their bins are found separately, because these borrowers and their default rates are different.

In [22]:
binning_60 = BinningProcess(
    variable_names=features,
    categorical_variables=categories,
    min_bin_size=0.05,
    binning_fit_params={c: {"monotonic_trend": "auto_asc_desc"} for c in features},
)
binning_60.fit(train_60[features], train_60["target"])

binning_60.summary()[["name", "dtype", "n_bins"]]

,name,dtype,n_bins
0,purpose,categorical,4
1,verification_status,categorical,3
2,home_ownership,categorical,3
3,annual_inc,numerical,7
4,dti,numerical,8
5,delinq_2yrs,numerical,2
6,inq_last_6mths,numerical,4
7,open_acc,numerical,4
8,total_acc,numerical,6
9,pub_rec,numerical,2


### 2.4 Saving the bins

The bins are saved so the next notebook can turn the train and test sets into WoE values with exactly the same bins.

In [23]:
binning_36.save(str(PROCESSED / "binning_36.pkl"))
binning_60.save(str(PROCESSED / "binning_60.pkl"))